In [12]:
import minsearch
import json
from openai import OpenAI
from dotenv import load_dotenv

In [13]:
load_dotenv()

True

In [2]:
with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

In [ ]:
# flatten the json data
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [6]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

In [7]:
index.fit(docs=documents)

In [15]:
q = "the course has already started, can I still enroll?"

In [ ]:
boost = { 'question': 3.0, 'section': .5 }

results=index.search(
    query=q,
    filter_dict={'course': 'data-engineering-zoomcamp'},
    boost_dict=boost,
    num_results=5
)

In [14]:
client = OpenAI()

In [16]:
response = client.responses.create(
    model="gpt-4o",
    input=[
        {
            "role": "user",
            "content": q
        }
    ]
)

print(response.output_text)

It's best to check with the institution or platform offering the course. Some may allow late enrollment, especially if it's within a certain time frame. Reach out to the course administrator or check their website for specific policies.


In [22]:
prompt_template = """
    You are a teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
    Use only the facts from the context when answering the QUESTION.
    If the context does not contain the answer, output NONE.

    QUESTION: {question}
    CONTEXT: {context}

""".strip()

In [23]:
context = ""

for doc in results:
    context += f"section {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"

In [26]:
prompt = prompt_template.format(question=q, context=context).strip()

In [27]:
prompt

"You are a teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.\n    Use only the facts from the context when answering the QUESTION.\n    If the context does not contain the answer, output NONE.\n\n    QUESTION: the course has already started, can I still enroll?\n    CONTEXT: section General course-related questions\nquestion: Course - Can I still join the course after the start date?\nanswer: Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.\n\nsection General course-related questions\nquestion: Course - Can I follow the course after it finishes?\nanswer: Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also st

In [28]:
response = client.responses.create(
    model="gpt-4o",
    input=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.output_text)

Yes, even if you don't register, you're still eligible to submit the homeworks. Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.
